# 01 - Bronze ingestion: Poloniex OHLCV hourly

**Objetivo:**
1. Buscar candles horários de OHLCV na API pública da Poloniex.
2. Coletar BTC, ETH, SOL, ADA e LINK.
3. Escrever a tabela Delta Bronze usando zordon.

**Estratégia de Ingestão:**
- **D-1 Incremental:** Pipeline coleta apenas o dia anterior (D-1) a cada execução
- **Cadência:** Scheduled para rodar 1x por dia
- **Idempotência:** Pode rodar múltiplas vezes no mesmo dia sem duplicar dados
- **Backfill:** Suporta reprocessamento de dias específicos alterando TARGET_DATE
- **Imutabilidade:** OHLCV candles são imutáveis após fechamento do período

**Destino:**  
`uc_sa_br_dev.bronze_poloniex_ohlcv.hourly`

**Princípios Bronze Layer (Pragmática):**
- Select relevant fields for the project (discard unnecessary ones)
- Keep original data types as-is (no type casting, no conversions)
- Add ONLY metadata for auditability: ingested_at
- Add rate_date ONLY for physical partitioning (storage optimization)
- Defer business transformations to Silver layer
- Source identification is implicit in schema name (bronze_poloniex_ohlcv)

**Campos Armazenados:**
- API Fields: symbol, low, high, open, close, volume, start_time_ms, close_time_ms, interval, trade_count (kept as original types)
- Audit Metadata: ingested_at (process timestamp)
- Physical Partition: rate_date (derived from start_time_ms for storage optimization only)
- Source: Implicit in schema name (bronze_poloniex_ohlcv) - NOT stored as field

**Benefícios da Estratégia D-1 Incremental:**
- **Performance:** ~95% menos dados escritos por execução (120 vs 2500 registros)
- **Custo:** API calls menores, menos compute time, menos transferência de dados
- **Idempotência:** Múltiplas execuções no mesmo dia não duplicam dados
- **Backfill Preciso:** Reprocessar dias específicos sem afetar histórico
- **Resiliência:** Proteção contra falhas com retry seguro

**Observação:**  
Business transformations (type casting to DECIMAL, timestamp conversion to TIMESTAMP, data quality validations) will be done in Silver.
The `exchange` field will be added in Silver as `F.lit('poloniex')` for multi-exchange analytics.
The daily aggregation will be derived in Gold from the hourly data.

Este arquivo deve ser executado no Databricks, onde `spark` já existe.

In [0]:
import json
import logging
import sys
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone


ZORDON_SRC_PATH = "/Workspace/Repos/CryptoLake/zordon-data-utils/src"

if ZORDON_SRC_PATH not in sys.path:
    sys.path.append(ZORDON_SRC_PATH)

import zordon
from pyspark.sql import functions as F

In [0]:
from datetime import date

SYMBOLS = ["BTC_USDT", "ETH_USDT", "SOL_USDT", "ADA_USDT", "LINK_USDT"]
INTERVAL = "HOUR_1"
TABLE_NAME = "hourly"
MAX_LIMIT = 500

# D-1 Incremental Strategy:
# Default: collect yesterday's data (D-1)
# For backfill: set TARGET_DATE to specific date (e.g., date(2026, 6, 15))
TARGET_DATE = date.today() - timedelta(days=1)

In [0]:
def utc_now_ms():
    """Timestamp atual em milissegundos UTC."""
    return int(datetime.now(tz=timezone.utc).timestamp() * 1000)


def utc_days_ago_ms(days):
    """Timestamp de N dias atras em milissegundos UTC."""
    dt = datetime.now(tz=timezone.utc) - timedelta(days=days)
    return int(dt.timestamp() * 1000)

In [0]:
def fetch_poloniex_candles(symbol, start_time_ms, end_time_ms, limit=MAX_LIMIT):
    """Busca candles horarios na API publica da Poloniex."""
    base_url = f"https://api.poloniex.com/markets/{symbol}/candles"
    params = urllib.parse.urlencode({
        "interval": INTERVAL,
        "startTime": start_time_ms,
        "endTime": end_time_ms,
        "limit": limit,
    })
    url = f"{base_url}?{params}"

    with urllib.request.urlopen(url, timeout=30) as response:
        if response.status != 200:
            raise RuntimeError(f"Erro na API Poloniex: HTTP {response.status}")
        payload = response.read().decode("utf-8")

    return json.loads(payload)


def parse_candle(symbol, row):
    """Extract relevant fields for CryptoLake project.

    Pragmatic Bronze approach:
    - Select only fields needed for OHLCV analysis
    - Keep original data types (no casting, no conversions)
    - Discard fields not used in the project (buyTakerAmount, weightedAverage, etc.)

    Poloniex API response (14 fields):
    [0] low, [1] high, [2] open, [3] close, [4] amount, [5] quantity,
    [6] buyTakerAmount, [7] buyTakerQuantity, [8] tradeCount, [9] ts,
    [10] weightedAverage, [11] interval, [12] startTime, [13] closeTime.
    """
    return {
        "symbol": symbol,
        "low": row[0],
        "high": row[1],
        "open": row[2],
        "close": row[3],
        "volume": row[5],
        "start_time_ms": row[12],
        "close_time_ms": row[13],
        "interval": row[11],
        "trade_count": row[8],
    }

In [0]:
def collect_hourly_candles(symbols, target_date):
    """Collect hourly candles for multiple symbols for a specific date.

    Args:
        symbols: List of symbol pairs (e.g., ["BTC_USDT", "ETH_USDT"])
        target_date: Date to collect data for (date object)

    Returns:
        List of raw candle records with metadata.

    Note:
        D-1 Incremental Strategy: Collects only 24 hourly candles per symbol.
        For backfill, pass a specific date. For normal operation, use D-1.
        Current limitation: MAX_LIMIT (500) is sufficient for 1-day collection.
    """
    # Calculate start and end timestamps for the target date (00:00 to 23:59:59 UTC)
    start_time_ms = int(
        datetime.combine(target_date, datetime.min.time(), timezone.utc).timestamp() * 1000
    )
    end_time_ms = int(
        datetime.combine(target_date, datetime.max.time(), timezone.utc).timestamp() * 1000
    )

    all_records = []

    for symbol in symbols:
        try:
            logging.info(f"Fetching {symbol} candles...")
            candles = fetch_poloniex_candles(
                symbol=symbol,
                start_time_ms=start_time_ms,
                end_time_ms=end_time_ms,
                limit=MAX_LIMIT,
            )

            for candle_row in candles:
                record = parse_candle(symbol, candle_row)
                all_records.append(record)

            logging.info(f"Collected {len(candles)} candles for {symbol}")

        except Exception as e:
            logging.error(f"Failed to fetch {symbol}: {e}")
            raise

    return all_records

In [0]:
# 1. Criar client Bronze via zordon.
proj = zordon.Project(
    spark=spark,
    country="br",
    region="sa",
    environment="dev",
)

bronze_poloniex = proj.client(
    layer="bronze",
    domain="poloniex",
    subdomain="ohlcv",
)

target_fqn = bronze_poloniex.governance.fqn(TABLE_NAME)
logging.info(f"Target table: {target_fqn}")

In [0]:
# 2. Validar nome da tabela com zordon.
valid = zordon.is_valid_name(TABLE_NAME)
violation = zordon.name_violation(TABLE_NAME)
logging.info(f"Table name '{TABLE_NAME}' valid: {valid} | Violation: {violation}")

In [0]:
# 3. Buscar dados da Poloniex.
logging.info(f"Starting D-1 incremental collection for date: {TARGET_DATE}")
logging.info(f"Symbols: {SYMBOLS}")
records = collect_hourly_candles(SYMBOLS, TARGET_DATE)
logging.info(f"Total records collected: {len(records)} ({len(records) // len(SYMBOLS)} per symbol)")

if not records:
    raise RuntimeError("No records returned from Poloniex API")

In [0]:
# 4. Criar DataFrame Spark.
# Add ingested_at timestamp and rate_date for physical partitioning.
df = (
    spark.createDataFrame(records)
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn(
        "rate_date",
        F.to_date(F.from_unixtime(F.col("start_time_ms") / 1000))
    )
)

total_rows = df.count()
logging.info(f"DataFrame created with {total_rows} records")
display(df.limit(10))

In [0]:
# 5. Escrever Delta Bronze com zordon.
# Dynamic partition overwrite: allows reprocessing specific days without losing history.
# rate_date is used for physical partitioning (storage optimization),
# not as a business transformation (Silver will do proper timestamp conversions).

partitions_in_df = df.select("rate_date").distinct().collect()
partition_dates = [row.rate_date for row in partitions_in_df]
logging.info(f"Writing {len(partition_dates)} partition(s): {partition_dates}")
logging.info(f"D-1 Incremental: Only these partitions will be overwritten, preserving historical data")

written_fqn = bronze_poloniex.write_table(
    df=df,
    table_name=TABLE_NAME,
    mode="overwrite",
    partition_cols=["rate_date"],
    dynamic_partition_overwrite=True,
)

logging.info(f"Bronze table written: {written_fqn}")
logging.info(f"Target partition: {TARGET_DATE}")

In [0]:
# 6. Ler de volta com zordon e validar contagens.
df_read = bronze_poloniex.read_table(TABLE_NAME)

logging.info("Validation: counts per symbol and date range")
display(
    df_read
    .groupBy("symbol")
    .agg(
        F.count("*").alias("rows"),
        F.min("rate_date").alias("min_date"),
        F.max("rate_date").alias("max_date"),
        F.min("ingested_at").alias("first_ingested"),
    )
    .orderBy("symbol")
)

logging.info("Sample records from Bronze table")
display(df_read.limit(10))

In [0]:
# 7. Conferir tabela criada no schema Bronze.
display(spark.sql("SHOW TABLES IN uc_sa_br_dev.bronze_poloniex_ohlcv"))

logging.info("Bronze Poloniex hourly ingestion completed successfully")

## Backfill: Como reprocessar dias específicos

A estratégia D-1 incremental permite reprocessar dias específicos sem perder histórico.

### Caso 1: Reprocessar um dia específico
```python
# Na célula 3, altere TARGET_DATE:
TARGET_DATE = date(2026, 6, 20)  # Reprocessar 20/06/2026

# Execute as células 9-11
# Resultado: apenas a partição 2026-06-20 será reescrita
```

### Caso 2: Backfill de múltiplos dias
```python
# Criar loop para processar vários dias
from datetime import date, timedelta

start_date = date(2026, 6, 15)
end_date = date(2026, 6, 20)

current_date = start_date
while current_date <= end_date:
    logging.info(f"Processing {current_date}")
    records = collect_hourly_candles(SYMBOLS, current_date)
    df = spark.createDataFrame(records).withColumn("ingested_at", F.current_timestamp()).withColumn("rate_date", F.lit(current_date))
    bronze_poloniex.write_table(df, TABLE_NAME, mode="overwrite", partition_cols=["rate_date"], dynamic_partition_overwrite=True)
    current_date += timedelta(days=1)
```

### Caso 3: Verificação de dados ausentes
```python
# Verificar quais dias estão na tabela
df_dates = bronze_poloniex.read_table(TABLE_NAME).select("rate_date").distinct().orderBy("rate_date")
display(df_dates)

# Se identificar buracos, reprocessar apenas os dias ausentes
```

In [0]:
%sql
drop table if exists uc_sa_br_dev.bronze_poloniex_ohlcv.hourly